# Ingesta Bronze de la IPS

Cargar los archivos fuente en tablas Delta dentro de `workspace.bronze`, conservando los datos originales y agregando metadatos de trazabilidad.

## 1. Setup — Dependencias e imports

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

## 2. Configuración de rutas de archivos fuente

In [0]:
# Se centralizan las rutas para evitar repetirlas en las llamadas de carga.
RUTA_BASE = "/Volumes/workspace/default/archivos_crudos"

RUTAS_ARCHIVOS = {
    "citas": f"{RUTA_BASE}/citas.xlsx",
    "pacientes": f"{RUTA_BASE}/pacientes.xlsx",
    "eventos_clinicos": f"{RUTA_BASE}/eventos_clinicos.xlsx",
    "facturacion": f"{RUTA_BASE}/facturacion.xlsx",
}

## 3. Función de lectura con trazabilidad

In [0]:
# Lee un Excel, adapta columnas "object" a string (evita conflictos de tipo)
# y agrega metadatos de trazabilidad (archivo origen, fecha de ingesta, lote).
def leer_excel_como_spark(ruta_archivo: str, nombre_archivo: str, lote_ingesta: str):
    try:
        df_pandas = pd.read_excel(ruta_archivo)

        columnas_pandas = list(df_pandas.columns)
        for columna in columnas_pandas:
            if df_pandas[columna].dtype == "object":
                df_pandas[columna] = df_pandas[columna].astype("string")

        df_spark = spark.createDataFrame(df_pandas)
    except Exception as e:
        print(f"Error al leer el archivo Excel: {e}")
        raise

    return df_spark.withColumns({
        "fuente_archivo": F.lit(nombre_archivo),
        "fecha_ingesta": F.current_timestamp(),
        "lote_ingesta": F.lit(lote_ingesta),
    })

## 4. Carga de las 4 fuentes

In [0]:
fecha_str = datetime.now().strftime("%Y_%m_%d")
lote_actual = f"lote#{fecha_str}"

df_citas_bronze = leer_excel_como_spark(RUTAS_ARCHIVOS["citas"], "citas.xlsx", lote_actual)
df_pacientes_bronze = leer_excel_como_spark(RUTAS_ARCHIVOS["pacientes"], "pacientes.xlsx", lote_actual)
df_eventos_bronze = leer_excel_como_spark(RUTAS_ARCHIVOS["eventos_clinicos"], "eventos_clinicos.xlsx", lote_actual)
df_facturacion_bronze = leer_excel_como_spark(RUTAS_ARCHIVOS["facturacion"], "facturacion.xlsx", lote_actual)

## 5. Verificación de registros cargados

In [0]:
# Se verifica que cada fuente haya sido leída y tenga el número esperado de registros antes de escribir las tablas.
print(f"citas: {df_citas_bronze.count()} filas")
print(f"pacientes: {df_pacientes_bronze.count()} filas")
print(f"eventos_clinicos: {df_eventos_bronze.count()} filas")
print(f"facturacion: {df_facturacion_bronze.count()} filas")

## 6. Escribir tablas Bronze

In [0]:
df_citas_bronze.write.mode("overwrite").format("delta").saveAsTable("workspace.bronze.citas")
df_pacientes_bronze.write.mode("overwrite").format("delta").saveAsTable("workspace.bronze.pacientes")
df_eventos_bronze.write.mode("overwrite").format("delta").saveAsTable("workspace.bronze.eventos_clinicos")
df_facturacion_bronze.write.mode("overwrite").format("delta").saveAsTable("workspace.bronze.facturacion")

print("Capa Bronze completa")

## Resultado

La capa Bronze quedó implementada con cuatro tablas Delta:

- `workspace.bronze.citas`
- `workspace.bronze.pacientes`
- `workspace.bronze.eventos_clinicos`
- `workspace.bronze.facturacion`

Las tablas conservan los datos recibidos desde las fuentes originales y contienen metadatos de trazabilidad.

Las transformaciones de calidad, tipificación y reglas de negocio se aplicarán en la capa Silver.